In [4]:
# ============================================================
# Last Mile Delivery - Data Generator
# ============================================================
# This script creates a fake (synthetic) delivery dataset
# that looks like real data from a logistics company.
# Each row = one delivery attempt.
# ============================================================

# --- Step 1: Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import pandas as pd

# Create the project folder in Drive if it doesn't exist
os.makedirs("/content/drive/MyDrive/last_mile_project/data", exist_ok=True)

# --- Step 2: Set a random seed so results are the same every time ---
np.random.seed(42)
N = 5000  # number of delivery records we want to generate


# --- Step 3: Generate each column one by one ---

# Delivery IDs: just a unique label for each row
delivery_ids = [f"DLV-{i:05d}" for i in range(1, N + 1)]

# Random dates throughout 2024
dates = pd.date_range(start="2024-01-01", end="2024-12-31", freq="D")
delivery_dates = np.random.choice(dates, size=N)

# City zone: where the delivery is going
# Urban areas are busier and harder to deliver to
zones = np.random.choice(
    ["urban_core", "urban_inner", "suburban", "rural"],
    size=N,
    p=[0.35, 0.25, 0.25, 0.15]  # probabilities must add up to 1
)

# Time window: when the customer asked for delivery
time_windows = np.random.choice(
    ["morning_08_12", "afternoon_12_17", "evening_17_21", "anytime"],
    size=N,
    p=[0.30, 0.40, 0.20, 0.10]
)

# Weather condition on the day of delivery
weather = np.random.choice(
    ["clear", "cloudy", "rain", "snow"],
    size=N,
    p=[0.55, 0.20, 0.18, 0.07]
)

# Attempt number: is this the 1st, 2nd, or 3rd try?
# Most deliveries succeed on the first attempt
attempt_number = np.random.choice([1, 2, 3], size=N, p=[0.72, 0.22, 0.06])

# Driver experience in years (most drivers are newer, few are veterans)
driver_experience = np.clip(np.random.exponential(scale=3.0, size=N), 0.5, 20.0).round(1)

# Number of packages in this delivery
package_count = np.random.randint(1, 12, size=N)

# Total weight (heavier packages take longer)
weight_kg = (np.random.uniform(0.3, 3.0, size=N) * package_count).round(2)

# How many stops the driver made BEFORE this delivery (fatigue factor)
stops_before = np.random.randint(0, 40, size=N)

# Traffic index from 1 (clear road) to 10 (gridlock)
# Urban zones have higher traffic on average
traffic_base = {"urban_core": 7.2, "urban_inner": 5.8, "suburban": 3.5, "rural": 1.8}
traffic_index = np.clip(
    [np.random.normal(traffic_base[z], 1.5) for z in zones],
    1.0, 10.0
).round(1)

# Distance from depot to delivery address in km
distance_base = {"urban_core": 4.5, "urban_inner": 8.0, "suburban": 14.0, "rural": 28.0}
distance_km = np.clip(
    [abs(np.random.normal(distance_base[z], 3.0)) for z in zones],
    0.5, 60.0
).round(2)


# --- Step 4: Calculate delivery outcome (success or failure) ---
# We build up a failure probability from multiple risk factors.
# The more risk factors stack up, the more likely the delivery fails.

failure_prob = np.zeros(N)

# Base failure rate by zone (urban is harder)
zone_base = {"urban_core": 0.18, "urban_inner": 0.13, "suburban": 0.09, "rural": 0.07}
failure_prob += np.array([zone_base[z] for z in zones])

# Time window risk (vague windows = customer not ready)
window_penalty = {"morning_08_12": -0.03, "afternoon_12_17": 0.05, "evening_17_21": -0.02, "anytime": 0.08}
failure_prob += np.array([window_penalty[tw] for tw in time_windows])

# Weather risk (snow is worst)
weather_penalty = {"clear": 0.00, "cloudy": 0.01, "rain": 0.05, "snow": 0.12}
failure_prob += np.array([weather_penalty[w] for w in weather])

# Each repeat attempt is harder (customer getting frustrated / less available)
failure_prob += (attempt_number - 1) * 0.07

# Less experienced drivers fail more often
failure_prob += np.clip(1.0 - (driver_experience / 25.0), 0.0, 0.15)

# Heavy traffic means late arrival, customer may have left
failure_prob += np.clip((traffic_index - 5) * 0.015, 0, 0.10)

# Tired driver (many stops already) = more errors
failure_prob += np.clip(stops_before * 0.003, 0, 0.12)

# Cap between 2% and 85% (never impossible, never certain)
failure_prob = np.clip(failure_prob, 0.02, 0.85)

# Roll the dice: 1 = delivery succeeded, 0 = failed
delivery_success = (np.random.random(N) > failure_prob).astype(int)


# --- Step 5: Calculate delay minutes (only for successful deliveries) ---
# Failed deliveries don't have a delay — they just didn't happen.

delay_minutes = np.random.normal(loc=8.0, scale=5.0, size=N)

# Weather makes deliveries slower
weather_delay = {"clear": 0, "cloudy": 3, "rain": 12, "snow": 25}
delay_minutes += np.array([weather_delay[w] for w in weather])

# Traffic slows everything down
delay_minutes += np.clip((traffic_index - 1) / 9.0, 0, 1) * 20

# Driver fatigue from previous stops
delay_minutes += stops_before * 0.3

delay_minutes = np.clip(delay_minutes, 0, 120).round(1)

# Set delay to NaN for failed deliveries (no delivery = no delay to measure)
delay_minutes = np.where(delivery_success == 1, delay_minutes, np.nan)


# --- Step 6: Put everything into a DataFrame ---

df = pd.DataFrame({
    "delivery_id":            delivery_ids,
    "date":                   delivery_dates,
    "day_of_week":            pd.to_datetime(delivery_dates).day_name(),
    "month":                  pd.to_datetime(delivery_dates).month,
    "city_zone":              zones,
    "time_window":            time_windows,
    "weather":                weather,
    "attempt_number":         attempt_number,
    "driver_experience_yrs":  driver_experience,
    "package_count":          package_count,
    "weight_kg":              weight_kg,
    "distance_km":            distance_km,
    "stops_before_this":      stops_before,
    "traffic_index":          traffic_index,
    "delivery_success":       delivery_success,
    "delay_minutes":          delay_minutes,
})

# --- Step 7: Save to Google Drive ---

output_path = "/content/drive/MyDrive/last_mile_project/data/last_mile_deliveries.csv"
df.to_csv(output_path, index=False)

print(f"✓ Saved {len(df):,} rows to Google Drive")
print(f"✓ Failure rate: {(1 - df['delivery_success'].mean()):.1%}")
print(f"✓ Avg delay (successful deliveries): {df['delay_minutes'].mean():.1f} min")
print("\nFirst 3 rows:")
df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Saved 5,000 rows to Google Drive
✓ Failure rate: 40.0%
✓ Avg delay (successful deliveries): 26.6 min

First 3 rows:


,delivery_id,date,day_of_week,month,city_zone,time_window,weather,attempt_number,driver_experience_yrs,package_count,weight_kg,distance_km,stops_before_this,traffic_index,delivery_success,delay_minutes
0,DLV-00001,2024-04-12,Friday,4,urban_inner,afternoon_12_17,cloudy,2,2.3,10,20.38,9.74,24,3.3,1,29.0
1,DLV-00002,2024-12-14,Saturday,12,urban_inner,afternoon_12_17,clear,1,3.2,1,0.75,3.87,33,5.0,0,NaN
2,DLV-00003,2024-09-27,Friday,9,rural,afternoon_12_17,rain,2,7.1,6,12.66,32.02,8,1.9,1,21.1
